In [ ]:
import os
import requests
import base64
import pandas as pd
from dotenv import load_dotenv

# === Load GitHub Token ===
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN_6")
HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.mercy-preview+json",
    "User-Agent": "android-verifier"
}

# === Configuration ===
TARGET_LANGUAGES = {"Java", "Kotlin", "Dart"}
KEYWORD = "android"

# === Helper Functions ===
def get_repo_metadata(full_name):
    url = f"https://api.github.com/repos/{full_name}"
    r = requests.get(url, headers=HEADERS)
    return r.json() if r.status_code == 200 else None

def get_repo_topics(full_name):
    url = f"https://api.github.com/repos/{full_name}/topics"
    r = requests.get(url, headers=HEADERS)
    return r.json().get("names", []) if r.status_code == 200 else []

def get_repo_readme(full_name):
    url = f"https://api.github.com/repos/{full_name}/readme"
    r = requests.get(url, headers=HEADERS)
    if r.status_code == 200:
        try:
            content = r.json().get("content", "")
            return base64.b64decode(content).decode('utf-8').lower()
        except Exception:
            return ""
    return ""

def validate_repo(full_name, initial_language, initial_fields):
    metadata = get_repo_metadata(full_name)
    if not metadata:
        return {"repo": full_name, "status": "metadata_fetch_failed"}

    actual_lang = metadata.get("language", "")
    if actual_lang not in TARGET_LANGUAGES:
        return {"repo": full_name, "initial_language": initial_language, "actual_language": actual_lang, "status": "language_rejected"}

    name = metadata.get("name", "").lower()
    desc = metadata.get("description", "").lower() if metadata.get("description") else ""

    found_fields = []
    topics = get_repo_topics(full_name)
    if any(KEYWORD in t.lower() for t in topics):
        found_fields.append("topic")
    if KEYWORD in name or KEYWORD in desc:
        found_fields.append("name/description")

    readme = get_repo_readme(full_name)
    if KEYWORD in readme:
        found_fields.append("readme")

    if not found_fields:
        return {"repo": full_name, "status": "keyword_missing"}

    updates = {}
    if actual_lang != initial_language and actual_lang in TARGET_LANGUAGES:
        updates["language"] = actual_lang

    for field in ["topic", "name/description", "readme"]:
        if field in found_fields and field not in initial_fields:
            updates[field] = True

    return {
        "repo": full_name,
        "initial_language": initial_language,
        "actual_language": actual_lang,
        "fields_found": found_fields,
        "updated_fields": updates,
        "status": "accepted"
    }

# === Example: List of Repos from GitHub Search ===
# Each repo has: full_name, language from search, and fields where 'android' was matched
sample_repos = [
    {"full_name": "google/flutter", "language": "Dart", "fields": ["topic"]},
    {"full_name": "octocat/Hello-World", "language": "Java", "fields": ["readme"]},
    {"full_name": "JetBrains/kotlin", "language": "Kotlin", "fields": ["name/description"]},
    {"full_name": "user/unknown-repo", "language": "Java", "fields": []},
]

# === Run Filtering ===
validated = []
rejected = []

for repo in sample_repos:
    result = validate_repo(repo["full_name"], repo["language"], repo["fields"])
    if result["status"] == "accepted":
        validated.append(result)
    else:
        rejected.append(result)

# === Save Results ===
pd.DataFrame(validated).to_csv("validated_repos.csv", index=False)
pd.DataFrame(rejected).to_csv("rejected_repos.csv", index=False)

print("✅ Validation completed. Results saved to CSV.")
